In [1]:
# ============================================================
# 037_Claude_Structure_Driven_Monitor_Notebook_Cloner (Skill-aware)
# ============================================================
#
# Overview
# ----------------
# This notebook generates new "monitor_*_daily" notebooks (e.g., 034/035) by cloning a
# reference notebook (e.g., 032_monitor_vc_daily.ipynb) with **minimal, domain-only edits**
# using a two-phase, structure-first authoring flow (similar to #026).
#
# Target notebooks share the same pipeline invariants:
# - Monitoring Targets read
# - Diff collection -> Event normalization -> dedup -> Events DB save
#
# Phase 1 (Structure as the source of truth):
# - User provides:
#     - SOURCE_NOTEBOOK (selected via UI widgets: upload or local pick)
#     - TARGET_DOMAIN (e.g., "startup", "policy", "people")
#     - OUT_NOTEBOOK path (e.g., 033_monitor_startup_daily.ipynb)
# - Claude generates ONLY Cell 00 of the target notebook, including an explicit "# Structure"
#   section enumerating the planned cells for the target notebook (Cell 01..N).
# - Python parses "# Structure" and deterministically builds a skeleton target notebook
#   (Cell 00 + empty Cell 01..N) using nbformat.
#
# Phase 2 (Incremental cell filling; minimal-diff cloning):
# - Step A (PLAN; light):
#     - Claude returns a per-cell decision for an allowlist: keep vs edit (no base64/code here).
# - Step B (BUILD; per-cell loop):
#     - keep: Python copies the source cell into the target notebook (byte-stable where possible).
#     - edit: Claude generates ONE cell replacement at a time (base64-only or strict JSON),
#       Python decodes + applies immediately, then moves to the next cell.
# - Step C (SAVE):
#     - Python clears outputs/execution_count and writes the final .ipynb to disk.
#
# Design principle (strict separation of concerns):
# - LLM proposes structure/content in constrained formats (optionally guided by Skills),
# - Python validates, parses, assembles, applies, verifies, and saves deterministically.
#
# NEW (Skill-aware authoring):
# - Loads "human correction" Skill records produced by #027 (Notion-synced JSONL preferred).
# - Selects a small, relevant subset (Skill Pack) based on:
#     - the user’s spec and/or target_domain
#     - the source notebook content
# - Injects Skill Pack into Claude prompts for:
#     - Cell 00 structure drafting
#     - Plan generation (keep/edit)
#     - Per-cell edit generation (one cell at a time)
#
# Inputs / Outputs
# ----------------
# Inputs:
# - env.txt (single entry point; never write keys into generated notebooks)
#   - ANTHROPIC_API_KEY (required for authoring)
#   - CLAUDE_MODEL (optional)
#   - OPENAI_API_KEY (optional; only if the generated target notebook uses OpenAI at runtime)
# - SKILLS_JSONL_PATH (optional; default: skills/human_corrections.jsonl)
# - UI widgets:
#   - SOURCE_NOTEBOOK selection (upload or local dropdown)
#   - TARGET_DOMAIN (startup/policy/people/...)
#   - OUT_NOTEBOOK path
#   - Preview toggles
#   - Skill controls (enable, top-k, filters)
#
# Outputs:
# - Skeleton target notebook (.ipynb), deterministic:
#   - Cell 00 (Claude-authored; includes "# Structure")
#   - Cell 01..N empty placeholders from parsed Structure list
# - Final patched notebook written to disk:
#   - e.g., 033_monitor_startup_daily.ipynb
#   - e.g., 034_monitor_policy_daily.ipynb
# - Optional Python-generated logs:
#   - plan summary (keep/edit indices)
#   - per-cell edit status (success/failure per index)
# - In-memory state:
#   - LAST_SOURCE_NOTEBOOK_PATH (optional; if selected from local)
#   - LAST_SOURCE_NOTEBOOK_OBJ
#   - LAST_TARGET_DOMAIN
#   - LAST_OUT_NOTEBOOK_PATH
#   - LAST_SPEC_TEXT (optional)
#   - LAST_SKILL_PACK
#   - LAST_CLAUDE_CELL00_JSON
#   - LAST_STRUCTURE_ITEMS
#   - LAST_SKELETON_NOTEBOOK_PATH
#   - LAST_PATCH_PLAN
#   - LAST_TARGET_NOTEBOOK_OBJ
#   - LAST_FINAL_NOTEBOOK_PATH
#
# Structure
# ----------------
# Cell 1: Imports & Environment
#   - Standard libs + nbformat + ipywidgets + requests (or anthropic SDK)
#   - Load env vars explicitly from env.txt
#
# Cell 1b: Skill Store Loader & Retriever (optional but recommended)
#   - Load Skills JSONL
#   - Retrieve top-k Skill Pack from spec/source/target_domain
#
# Cell 2: Claude Cell 00 Writer (Skill-aware)
#   - Strict JSON schema: { "notebook_id": str, "cell00_source": str }
#   - Claude generates ONLY Cell 00, must include "# Structure" list
#
# Cell 3: Structure Parser & Skeleton Assembler (nbformat)
#   - Parse "# Structure" into [(cell_index, title), ...]
#   - Build skeleton notebook: Cell 00 + empty cells 01..N
#   - Save skeleton + preview
#
# Cell 4: UI (ipywidgets)  ※ intended to run last
#   - Inputs:
#       - Source notebook selection (Upload / Local)
#       - target domain / out path / spec / preview / skills
#   - Buttons:
#       - Phase 1: Generate skeleton
#       - Phase 2: Plan (keep/edit)
#       - Phase 2: Build (per-cell loop)
#       - Phase 2: Save
#
# Cell 5: Phase 1 Orchestrator (Generate Skeleton)
#   - Validate inputs
#   - (optional) retrieve Skill Pack
#   - call Claude to produce Cell 00 (with Structure)
#   - parse Structure and assemble/save skeleton via nbformat
#
# Cell 6a: Phase 2 Planner + Builder (per-cell loop)
#   - Plan: keep/edit decisions for allowlist (no base64)
#   - Build: keep=python copy, edit=Claude per-cell generation (one cell at a time)
#
# Cell 6b: Save / Verify utilities (optional but recommended)
#   - Clear outputs/execution_count
#   - Optional invariants verification (cell count/type)
#   - Save final notebook
#
# Notes
# ----------------
# - Reliability:
#   "# Structure" in Cell 00 is the single source of truth for cell ordering/count.
#
# - Minimal-diff philosophy:
#   The closer targets stay to the source, the easier maintenance becomes.
#
# - Security:
#   Never embed API keys into generated notebooks; env.txt is the only config entry point.
#
# - Extensibility:
#   Skills improve output over time; re-run #027 to accumulate corrections and reuse them here.


In [2]:
# ============================================================
# Cell 1 — Imports & Environment
# ============================================================
# Overview:
# - Import standard libraries used across Phase 1 / Phase 2.
# - Load environment variables explicitly from env.txt.
# - Initialize global, long-lived state containers.
# - NO Claude / OpenAI calls are performed in this cell.
#
# Inputs / Outputs:
# Inputs:
# - env.txt (optional)
#   - ANTHROPIC_API_KEY
#   - CLAUDE_MODEL
#   - OPENAI_API_KEY (optional; runtime-only for generated notebooks)
#
# Outputs:
# - Loaded environment variables in os.environ
# - Initialized global state placeholders (LAST_*)
#
# Notes:
# - This cell must be idempotent.
# - No notebook I/O or widget logic here.
# - Downstream cells rely on globals defined here.
# ============================================================

from __future__ import annotations

import os
import json
import re
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import nbformat
from nbformat.v4 import new_notebook, new_markdown_cell, new_code_cell

# ------------------------------------------------------------
# Environment loader
# ------------------------------------------------------------
def load_env_txt(env_path: Path) -> None:
    """
    Load KEY=VALUE lines from env.txt into os.environ.
    Existing environment variables are NOT overwritten.
    """
    if not env_path.exists():
        return

    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)


# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------
PROJECT_DIR = Path(".").resolve()
ENV_TXT_PATH = PROJECT_DIR / "env.txt"

load_env_txt(ENV_TXT_PATH)

# ------------------------------------------------------------
# Required / optional environment variables
# ------------------------------------------------------------
ANTHROPIC_API_KEY: str = os.environ.get("ANTHROPIC_API_KEY", "")
CLAUDE_MODEL: str = os.environ.get(
    "CLAUDE_MODEL",
    "claude-3-5-sonnet-20240620",
)

OPENAI_API_KEY: Optional[str] = os.environ.get("OPENAI_API_KEY")

# ------------------------------------------------------------
# Global in-memory state (shared across cells)
# ------------------------------------------------------------
LAST_SOURCE_NOTEBOOK_PATH: Optional[Path] = None
LAST_SOURCE_NOTEBOOK_OBJ: Optional[dict] = None

LAST_TARGET_DOMAIN: Optional[str] = None
LAST_OUT_NOTEBOOK_PATH: Optional[Path] = None

LAST_SPEC_TEXT: Optional[str] = None

LAST_SKILL_PACK: Optional[list] = None

LAST_CLAUDE_CELL00_JSON: Optional[dict] = None
LAST_STRUCTURE_ITEMS: Optional[list] = None
LAST_SKELETON_NOTEBOOK_PATH: Optional[Path] = None

LAST_PATCH_JSON: Optional[dict] = None
LAST_FINAL_NOTEBOOK_PATH: Optional[Path] = None
LAST_PATCH_LOG: Optional[str] = None

# ------------------------------------------------------------
# Notebook I/O helpers (required by Cell 4 UI)
# ------------------------------------------------------------
def read_ipynb_from_path(path: Path) -> dict:
    obj = nbformat.read(str(path), as_version=4)
    return json.loads(json.dumps(obj))  # ensure pure-JSON dict

def read_ipynb_from_bytes(data: bytes) -> dict:
    obj = nbformat.reads(data.decode("utf-8"), as_version=4)
    return json.loads(json.dumps(obj))

def write_ipynb(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nbformat.from_dict(obj), str(path))

import traceback
from datetime import datetime

def _pp_banner(title: str):
    log_anywhere("="*80)
    log_anywhere(title)
    log_anywhere("="*80)

def _pp_kv(k: str, v: str):
    log_anywhere(f"- {k}: {v}")

def _pp_exc(e: Exception, *, context: str = ""):
    _pp_banner(f"❌ FAIL {context}")
    _pp_kv("exception_type", type(e).__name__)
    _pp_kv("exception_msg", str(e))
    tb = traceback.format_exc().splitlines()
    log_anywhere("[traceback tail]")
    for line in tb[-15:]:
        log_anywhere(line)

def _pp_ok(context: str, extra: str = ""):
    _pp_banner(f"✅ SUCCESS {context}")
    if extra:
        for line in extra.splitlines():
            log_anywhere(line)

# ------------------------------------------------------------
# Basic sanity checks (non-fatal)
# ------------------------------------------------------------
if not ANTHROPIC_API_KEY:
    print("[WARN] ANTHROPIC_API_KEY is not set. Phase 1/2 authoring will fail.")

In [3]:
# ============================================================
# Cell 1b — Skill Store Loader & Retriever (optional but recommended)
# ============================================================
# Overview:
# - Load Skill records produced by #027 (Notion-synced JSON/JSONL).
# - Normalize records into a stable internal schema:
#     - content: str  (derived primarily from new_approach + reasoning + title)
#     - tags:    list[str] (correction_type + reason_tags + optional status)
#     - keywords:list[str] (lightweight: title tokens + optional hints)
# - Provide deterministic, debuggable Skill Pack retrieval (keyword/tag match).
#
# Inputs / Outputs:
# Inputs:
# - SKILLS_JSONL_PATH (env or default):
#     - Default: skills/human_corrections.jsonl
# - LAST_SPEC_TEXT (optional)
# - LAST_SOURCE_NOTEBOOK_OBJ (optional)
# - LAST_TARGET_DOMAIN (optional)
#
# Outputs:
# - SKILL_STORE: list[dict] (normalized)
# - retrieve_skill_pack(...)
# - Updates: LAST_SKILL_PACK
#
# Notes:
# - Your file appears to be a JSON array of objects (root: [ {..}, ... ]).
#   This loader supports both JSON array and JSONL.
# ============================================================

from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
import json
import re

SKILLS_JSONL_PATH: Path = Path(os.environ.get("SKILLS_JSONL_PATH", "skills/human_corrections.jsonl"))

# ----------------------------
# Normalization (tailored to #027 export schema)
# ----------------------------
def _as_str(x: Any) -> str:
    if x is None:
        return ""
    return str(x)

def _normalize_list(v: Any) -> List[str]:
    if v is None:
        return []
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    if isinstance(v, str):
        # allow comma-separated
        parts = [p.strip() for p in v.split(",")]
        return [p for p in parts if p]
    s = str(v).strip()
    return [s] if s else []

def _tokenize_title(title: str) -> List[str]:
    """
    Super lightweight keyword extraction from title.
    (No NLP deps; deterministic.)
    """
    t = title.lower()
    # split on non-alphanum, keep reasonably-long tokens
    toks = re.split(r"[^a-z0-9_]+", t)
    toks = [x for x in toks if len(x) >= 4]
    # de-dupe while keeping order
    seen = set()
    out = []
    for x in toks:
        if x not in seen:
            seen.add(x)
            out.append(x)
    return out

def normalize_skill_record_027(obj: Dict[str, Any]) -> Optional[Dict[str, Any]]:
    """
    Normalize a #027-style "human correction" record into:
      {
        id: str
        content: str
        tags: list[str]
        keywords: list[str]
        ... (original fields retained)
      }
    """
    title = _as_str(obj.get("title"))
    new_approach = _as_str(obj.get("new_approach"))
    reasoning = _as_str(obj.get("reasoning"))
    correction_type = _as_str(obj.get("correction_type"))
    reason_tags = _normalize_list(obj.get("reason_tags"))
    status = _as_str(obj.get("status"))

    # Core: if new_approach is missing, this is likely not usable as a "skill"
    # (we can relax this later if needed)
    if not new_approach and not reasoning and not title:
        return None

    # Build "content" (the thing we inject into prompts)
    parts = []
    if title:
        parts.append(f"[Title] {title}")
    if correction_type:
        parts.append(f"[Type] {correction_type}")
    if new_approach:
        parts.append(f"[New Approach]\n{new_approach}")
    if reasoning:
        parts.append(f"[Reasoning]\n{reasoning}")

    content = "\n".join(parts).strip()
    if not content:
        return None

    # Tags: correction_type + reason_tags + (optional) status
    tags: List[str] = []
    if correction_type:
        tags.append(correction_type)
    tags.extend(reason_tags)
    if status:
        tags.append(f"status:{status}")

    # Keywords: title tokens + a few stable fields
    keywords: List[str] = []
    keywords.extend(_tokenize_title(title))
    if correction_type:
        keywords.append(correction_type.lower())

    # id preference order
    sid = obj.get("skill_id") or obj.get("id") or obj.get("uuid") or obj.get("name") or ""
    sid = str(sid)

    out = dict(obj)  # keep raw fields (debuggable)
    out["id"] = sid
    out["content"] = content
    out["tags"] = [t for t in tags if t]
    out["keywords"] = [k for k in keywords if k]
    return out

# ----------------------------
# Loader (JSON array OR JSONL)
# ----------------------------
def load_skill_store_robust(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        print(f"[INFO] Skill store not found: {path}")
        return []

    raw_text = path.read_text(encoding="utf-8").strip()
    if not raw_text:
        print(f"[INFO] Skill store is empty: {path}")
        return []

    skills: List[Dict[str, Any]] = []

    # Case A: whole-file JSON array (matches your screenshot)
    if raw_text.startswith("["):
        try:
            data = json.loads(raw_text)
            if isinstance(data, list):
                for idx, obj in enumerate(data, start=1):
                    if not isinstance(obj, dict):
                        print(f"[WARN] Invalid skill entry (not dict) at idx {idx}")
                        continue
                    norm = normalize_skill_record_027(obj)
                    if norm is None:
                        print(f"[WARN] Invalid skill entry at idx {idx}: missing usable fields (need at least title/new_approach/reasoning)")
                        continue
                    skills.append(norm)
                print(f"[INFO] Loaded {len(skills)} skill records from JSON array: {path}")
                return skills
        except Exception as e:
            print(f"[WARN] Failed to parse JSON array from {path}: {e}")
            # fallthrough to JSONL

    # Case B: JSONL (one JSON object per line)
    for line_no, line in enumerate(raw_text.splitlines(), start=1):
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            if not isinstance(obj, dict):
                raise ValueError("not a JSON object")
            norm = normalize_skill_record_027(obj)
            if norm is None:
                raise ValueError("missing usable fields (need at least title/new_approach/reasoning)")
            skills.append(norm)
        except Exception as e:
            print(f"[WARN] Invalid skill record at line {line_no}: {e}")

    print(f"[INFO] Loaded {len(skills)} skill records from JSONL: {path}")
    return skills

SKILL_STORE: List[Dict[str, Any]] = load_skill_store_robust(SKILLS_JSONL_PATH)

# ----------------------------
# Retrieval (deterministic, debuggable)
# ----------------------------
def _score_skill(
    skill: Dict[str, Any],
    *,
    spec_text: Optional[str],
    target_domain: Optional[str],
    source_text: Optional[str],
) -> int:
    score = 0
    combined = " ".join([
        (spec_text or "").lower(),
        (target_domain or "").lower(),
        (source_text or "").lower(),
    ])

    # keywords (stronger)
    for kw in skill.get("keywords", []):
        if kw and kw.lower() in combined:
            score += 2

    # tags (weaker)
    for tag in skill.get("tags", []):
        if tag and tag.lower() in combined:
            score += 1

    return score

def retrieve_skill_pack(
    *,
    spec_text: Optional[str] = None,
    source_notebook_obj: Optional[dict] = None,
    target_domain: Optional[str] = None,
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    if not SKILL_STORE or top_k <= 0:
        return []

    source_text: Optional[str] = None
    if source_notebook_obj:
        source_text = "".join("".join(c.get("source", [])) for c in source_notebook_obj.get("cells", []))

    scored: List[Tuple[int, Dict[str, Any]]] = []
    for skill in SKILL_STORE:
        s = _score_skill(skill, spec_text=spec_text, target_domain=target_domain, source_text=source_text)
        if s > 0:
            scored.append((s, skill))

    scored.sort(key=lambda x: x[0], reverse=True)
    selected = [skill for _, skill in scored[:top_k]]

    global LAST_SKILL_PACK
    LAST_SKILL_PACK = selected
    return selected

def summarize_skill_pack(skills: List[Dict[str, Any]], max_chars: int = 140) -> None:
    if not skills:
        print("[INFO] Skill Pack is empty.")
        return
    print(f"[INFO] Skill Pack ({len(skills)} skills):")
    for i, s in enumerate(skills, start=1):
        sid = s.get("id") or f"skill_{i}"
        tags = ", ".join(s.get("tags", []))
        snippet = (s.get("content", "") or "").replace("\n", " ")
        if len(snippet) > max_chars:
            snippet = snippet[: max_chars - 3] + "..."
        print(f"  {i}. {sid}  [{tags}]  {snippet}")


[INFO] Loaded 40 skill records from JSONL: skills/human_corrections.jsonl


In [4]:
# ============================================================
# Cell 2 — Claude Cell 00 Writer (Skill-aware, Code-Header Style)
# ============================================================
# Overview:
# - Call Claude to generate ONLY Cell 00 for the target notebook as a CODE CELL SOURCE STRING.
# - Output STRICT JSON: { "notebook_id": str, "cell00_source": str }
#
# Inputs / Outputs:
# Inputs:
# - ANTHROPIC_API_KEY, CLAUDE_MODEL (env)
# - LAST_TARGET_DOMAIN, LAST_SPEC_TEXT, LAST_SOURCE_NOTEBOOK_OBJ, LAST_SKILL_PACK (optional)
#
# Outputs:
# - LAST_CLAUDE_CELL00_JSON
#
# Notes:
# - Cell 00 is CODE (comment header), not markdown.
# - Claude must return JSON ONLY.
# ============================================================

import requests
import uuid

def strip_code_fences_maybe(text: str) -> str:
    """
    If Claude wraps JSON in ```json ... ```, strip the fences.
    """
    t = text.strip()

    # Typical forms:
    # ```json\n{...}\n```
    # ```\n{...}\n```
    if t.startswith("```"):
        # remove first line (``` or ```json)
        lines = t.splitlines()
        if len(lines) >= 2 and lines[0].strip().startswith("```"):
            # find last fence
            if lines[-1].strip() == "```":
                t = "\n".join(lines[1:-1]).strip()
            else:
                # sometimes model forgets closing fence; just drop first line
                t = "\n".join(lines[1:]).strip()
    return t


def call_claude_messages_api(
    *,
    system_prompt: str,
    user_prompt: str,
    model: str,
    api_key: str,
    max_tokens: int = 20000,
    temperature: float = 0.2,
    timeout_s: int = 600,          # ★ read timeout を伸ばす
    max_retries: int = 3,          # ★ リトライ回数
    retry_backoff_s: float = 2.0,  # ★ 2,4,8... 秒
) -> str:
    """
    Returns assistant text (string).
    Retries on ReadTimeout / 5xx / 429.
    """
    url = "https://api.anthropic.com/v1/messages"
    headers = {
        "x-api-key": api_key,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json",
    }
    payload = {
        "model": model,
        "max_tokens": int(max_tokens),
        "temperature": float(temperature),
        "system": system_prompt,
        "messages": [{"role": "user", "content": user_prompt}],
    }

    last_err = None
    for attempt in range(1, int(max_retries) + 1):
        try:
            # timeout=(connect_timeout, read_timeout)
            resp = requests.post(url, headers=headers, json=payload, timeout=(30, timeout_s))
            # Retry対象: 429, 5xx
            if resp.status_code in (429,) or (500 <= resp.status_code <= 599):
                raise requests.HTTPError(f"HTTP {resp.status_code}: {resp.text[:300]}", response=resp)

            resp.raise_for_status()
            data = resp.json()

            # Claude messages API response shape:
            # data["content"] is list of blocks; take text blocks and concat.
            blocks = data.get("content", [])
            texts = []
            for b in blocks:
                if isinstance(b, dict) and b.get("type") == "text":
                    texts.append(b.get("text", ""))
            return "".join(texts).strip()

        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectTimeout) as e:
            last_err = e
            if "_log" in globals():
                _log(f"Claude API timeout (attempt {attempt}/{max_retries}). Retrying...", level="WARN")
            if attempt < max_retries:
                time.sleep(retry_backoff_s ** attempt)
                continue
            raise

        except requests.HTTPError as e:
            last_err = e
            # 429/5xx は上で例外化済み。ここでは再試行。
            if "_log" in globals():
                _log(f"Claude API HTTPError (attempt {attempt}/{max_retries}): {str(e)[:200]} ... Retrying...", level="WARN")
            if attempt < max_retries:
                time.sleep(retry_backoff_s ** attempt)
                continue
            raise

        except Exception as e:
            # それ以外は基本即死（必要ならここもリトライにしてOK）
            last_err = e
            raise

    # should never reach
    raise RuntimeError(f"Claude call failed: {type(last_err).__name__}: {last_err}")

def render_skill_pack_for_prompt(skills: Optional[List[Dict[str, Any]]], max_items: int = 8) -> str:
    if not skills:
        return "(no skills)"
    lines = []
    for s in skills[:max_items]:
        sid = s.get("id", "")
        content = (s.get("content", "") or "").strip()
        if len(content) > 900:
            content = content[:900] + "..."
        lines.append(f"- Skill {sid}:\n{content}")
    return "\n".join(lines).strip()

def build_cell00_system_prompt_code_header() -> str:
    return """
You are a senior Python/Jupyter notebook author.

You must output STRICT JSON ONLY (no markdown, no backticks, no commentary).
Schema:
{
  "notebook_id": "<string>",
  "cell00_source": "<string>"
}

cell00_source requirements:
- A Python CODE CELL source string.
- Contains only comments (and optionally a single 'pass' line).
- Must include this exact layout as comments:

# ============================================================
# <NOTEBOOK_ID_OR_NAME>
# ============================================================
#
# Overview
# ----------------
#
# Inputs / Outputs
# ----------------
#
# Structure
# ----------------
# Cell 01: ...
# Cell 02: ...
#
# Notes
# ----------------

Hard constraints:
- Under "Structure", list planned cells starting at "Cell 01:".
- Do NOT include executable logic, imports, or secrets.
""".strip()

def build_cell00_user_prompt_code_header(
    *,
    target_domain: str,
    out_notebook_name: str,
    spec_text: Optional[str],
    source_notebook_obj: Optional[dict],
    skill_pack: Optional[List[Dict[str, Any]]],
) -> str:
    notebook_header_name = Path(out_notebook_name).stem
    skills_text = render_skill_pack_for_prompt(skill_pack)
    spec_block = (spec_text or "").strip()

    if not spec_block:
        spec_block = f"""
Clone the source notebook with minimal, domain-only edits.
TARGET_DOMAIN = {target_domain}

Rules:
- Keep the pipeline identical.
- Only modify domain-specific parts.
- Do not refactor or reorder cells.

Allowed changes:
- Monitoring Targets query/filter
- Event normalization tags and entity type
- Notebook title and output filename
""".strip()

    # light hint from source (titles only)
    source_hint = ""
    if source_notebook_obj:
        titles = []
        for c in source_notebook_obj.get("cells", []):
            if c.get("cell_type") != "markdown":
                continue
            txt = "".join(c.get("source", []))
            if txt.strip().startswith("#"):
                titles.append(txt.strip().splitlines()[0].lstrip("#").strip())
        if titles:
            source_hint = "SOURCE NOTEBOOK HINT (markdown titles):\n- " + "\n- ".join(titles[:15])

    return f"""
TASK:
Write Cell 00 as a Python CODE CELL comment header for "{out_notebook_name}".

Format (must match):
# ============================================================
# {notebook_header_name}
# ============================================================
#
# Overview
# ----------------
#
# Inputs / Outputs
# ----------------
#
# Structure
# ----------------
# Cell 01: ...
# Cell 02: ...
#
# Notes
# ----------------

SKILL PACK:
{skills_text}

SPEC:
{spec_block}

{source_hint}
""".strip()


def validate_cell00_code_header(cell00_source: str) -> None:
    required = [
        "# ============================================================",
        "# Overview",
        "# Inputs / Outputs",
        "# Structure",
        "# Notes",
        "Cell 01:",
    ]
    missing = [m for m in required if m not in cell00_source]
    if missing:
        raise RuntimeError(f"Cell 00 missing required markers: {missing}")

    non_comment = []
    for line in cell00_source.splitlines():
        s = line.strip()
        if not s:
            continue
        if s.startswith("#"):
            continue
        if s == "pass":
            continue
        non_comment.append(line)

    if non_comment:
        raise RuntimeError("Cell 00 must be comment-only (optionally 'pass'). Found:\n" + "\n".join(non_comment[:20]))

def generate_cell00_json(
    *,
    target_domain: str,
    out_notebook_name: str,
    spec_text: Optional[str] = None,
    source_notebook_obj: Optional[dict] = None,   # <- 追加
    skill_pack: Optional[List[Dict[str, Any]]] = None,
    max_tokens: int = 20000,
) -> Dict[str, Any]:
    system_prompt = build_cell00_system_prompt_code_header()
    user_prompt = build_cell00_user_prompt_code_header(
        target_domain=target_domain,
        out_notebook_name=out_notebook_name,
        spec_text=spec_text,
        source_notebook_obj=source_notebook_obj,   # <- 追加
        skill_pack=skill_pack,
    )

    raw = call_claude_messages_api(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        model=CLAUDE_MODEL,
        api_key=ANTHROPIC_API_KEY,
        max_tokens=max_tokens,
        temperature=0.2,
    )

    raw_clean = strip_code_fences_maybe(raw)
    
    try:
        obj = json.loads(raw_clean)
    except Exception as e:
        raise RuntimeError(
            f"Claude did not return valid JSON: {e}\nRAW:\n{raw[:2000]}"
        )


    if "cell00_source" not in obj:
        raise RuntimeError(f"Claude JSON missing cell00_source. Keys={list(obj.keys())}")

    if not obj.get("notebook_id"):
        obj["notebook_id"] = str(uuid.uuid4())

    validate_cell00_code_header(obj["cell00_source"])
    return obj


print("[INFO] Cell 2 loaded: call_claude_messages_api + generate_cell00_json are ready.")


[INFO] Cell 2 loaded: call_claude_messages_api + generate_cell00_json are ready.


In [5]:
# ============================================================
# Cell 3 — Structure Parser & Skeleton Assembler (nbformat)
# ============================================================
# Overview:
# - Parse the "# Structure" section from Cell 00 (CODE cell comment header).
# - Deterministically assemble a skeleton .ipynb using nbformat:
#     - Cell 00: code cell (the generated header)
#     - Cell 01..N: empty code cells, each starting with the standard per-cell header:
#         # ============================================================
#         # Cell XX — <Title>
#         # ============================================================
#         # Overview:
#         #
#         # Inputs / Outputs:
#         #
#         # Notes:
# - Save the skeleton notebook to disk (path provided by orchestrator/UI).
# - Provide optional preview helpers.
#
# Inputs / Outputs:
# Inputs:
# - LAST_CLAUDE_CELL00_JSON (from Cell 2):
#     { "notebook_id": str, "cell00_source": str }
# - OUT_NOTEBOOK_PATH (passed explicitly) or LAST_OUT_NOTEBOOK_PATH
#
# Outputs:
# - LAST_STRUCTURE_ITEMS: list[tuple[int, str]]
#     e.g., [(1, "Imports & Environment"), (2, "Claude Cell 00 Writer"), ...]
# - LAST_SKELETON_NOTEBOOK_PATH: Path
# - Returns the assembled notebook dict (nbformat v4)
#
# Notes:
# - Parsing is strict and fail-fast: Structure must contain "Cell 01:" lines.
# - Assembly is deterministic: no model involvement here.
# - Outputs/execution counts are always cleared.
# ============================================================

from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import nbformat
from nbformat.v4 import new_notebook, new_code_cell
import re

# ----------------------------
# Parsing
# ----------------------------
STRUCTURE_SECTION_HEADER = "# Structure"
CELL_LINE_RE = re.compile(r"^\s*#\s*Cell\s+(\d{2})\s*:\s*(.+?)\s*$")

def parse_structure_items_from_cell00(cell00_source: str) -> List[Tuple[int, str]]:
    """
    Extract Structure items from a code-cell comment header.

    Expected lines under "# Structure" section:
    # Cell 01: <Title>
    # Cell 02: <Title>
    ...
    """
    lines = cell00_source.splitlines()

    # Locate "# Structure"
    start_idx = None
    for i, line in enumerate(lines):
        if line.strip() == STRUCTURE_SECTION_HEADER:
            start_idx = i + 1
            break
    if start_idx is None:
        raise RuntimeError('Cell 00 missing required section header: "# Structure"')

    items: List[Tuple[int, str]] = []
    for line in lines[start_idx:]:
        # Stop when next main section starts (e.g., "# Notes")
        if line.strip().startswith("# Notes"):
            break

        m = CELL_LINE_RE.match(line)
        if m:
            idx = int(m.group(1))
            title = m.group(2).strip()
            items.append((idx, title))

    if not items:
        raise RuntimeError('No structure items found. Expected lines like "# Cell 01: ...".')

    # Basic sanity: must start at 1 and be consecutive (strict mode)
    expected = 1
    for idx, _ in items:
        if idx != expected:
            raise RuntimeError(f"Structure cell indices must be consecutive starting at 01. Expected {expected:02d}, got {idx:02d}.")
        expected += 1

    return items


# ----------------------------
# Skeleton assembly
# ----------------------------
def make_cell_header(cell_index: int, title: str) -> str:
    """
    Build the standard per-cell header for Cell 01..N (code cell).
    """
    return (
        "# ============================================================\n"
        f"# Cell {cell_index:02d} — {title}\n"
        "# ============================================================\n"
        "# Overview:\n"
        "#\n"
        "# Inputs / Outputs:\n"
        "#\n"
        "# Notes:\n"
        "#\n"
    )

def strip_execution_outputs(nb: Dict[str, Any]) -> Dict[str, Any]:
    out = json.loads(json.dumps(nb))
    for c in out.get("cells", []):
        if c.get("cell_type") == "code":
            c["execution_count"] = None
            c["outputs"] = []
    return out

def assemble_skeleton_notebook(
    *,
    cell00_source: str,
    structure_items: List[Tuple[int, str]],
    notebook_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    """
    Assemble a deterministic nbformat v4 notebook object:
    - Cell 00 (code): header + optional 'pass'
    - Cells 01..N (code): header skeletons
    """
    nb = new_notebook()
    nb.metadata = notebook_metadata or {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python"},
    }

    # Cell 00: must remain exactly what Claude produced (plus optional trailing newline normalization)
    cell00_src = cell00_source.rstrip() + "\n"
    nb.cells.append(new_code_cell(source=cell00_src))

    # Cell 01..N: empty code cells with header template
    for idx, title in structure_items:
        header = make_cell_header(idx, title)
        nb.cells.append(new_code_cell(source=header))

    nb_dict = json.loads(json.dumps(nb))
    nb_dict = strip_execution_outputs(nb_dict)
    return nb_dict


# ----------------------------
# Save + preview helpers
# ----------------------------
def save_notebook(nb_obj: Dict[str, Any], out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nbformat.from_dict(nb_obj), str(out_path))

def preview_structure(structure_items: List[Tuple[int, str]], max_items: int = 50) -> None:
    print("[INFO] Parsed Structure:")
    for idx, title in structure_items[:max_items]:
        print(f"  Cell {idx:02d}: {title}")

def preview_cell_sources(nb_obj: Dict[str, Any], n_cells: int = 3, max_chars: int = 600) -> None:
    print(f"[INFO] Preview first {n_cells} cells:")
    for i, c in enumerate(nb_obj.get("cells", [])[:n_cells]):
        src = "".join(c.get("source", []))
        print(f"\n--- Cell {i:02d} ({c.get('cell_type')}) ---")
        print(src[:max_chars] + ("..." if len(src) > max_chars else ""))


# ----------------------------
# Orchestrator helper (Phase 1 assembly only)
# ----------------------------
from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import nbformat
import copy
import re

def _cell_source_str(cell: dict) -> str:
    src = cell.get("source", "")
    if isinstance(src, list):
        return "".join(src)
    return str(src)

def _make_empty_code_cell_with_header(cell_index: int, title: str = "") -> dict:
    header = (
        "# ============================================================\n"
        f"# Cell {cell_index:02d} — {title}\n"
        "# ============================================================\n"
        "# Overview:\n#\n"
        "# Inputs / Outputs:\n#\n"
        "# Notes:\n"
    )
    return nbformat.v4.new_code_cell(source=header)

def build_and_save_skeleton_from_cell00(
    *,
    cell00_json: Dict[str, Any],
    out_path: Path,
    do_preview: bool = True,
    source_notebook_obj: Optional[dict] = None,
) -> dict:
    """
    Build skeleton notebook deterministically.

    IMPORTANT (cloner mode):
    - If source_notebook_obj is provided, we FORCE the skeleton to match the source:
        - same cell count
        - same cell types per index
        - Cell 00 replaced by Claude's code header
        - Cells 01..N are empty placeholders BUT keep the same cell_type as source
          (code stays code, markdown stays markdown).
    - We do NOT rely on Claude "# Structure" for cell count.

    Writes notebook to out_path and updates globals:
      LAST_STRUCTURE_ITEMS, LAST_SKELETON_NOTEBOOK_PATH
    """
    out_path = Path(out_path)

    if "cell00_source" not in cell00_json:
        raise RuntimeError("cell00_json missing 'cell00_source'")

    cell00_src = cell00_json["cell00_source"]
    nb = nbformat.v4.new_notebook()

    # --- Determine target shape from source if available ---
    if source_notebook_obj is not None:
        src_cells = source_notebook_obj.get("cells", [])
        if not src_cells:
            raise RuntimeError("source_notebook_obj has no cells")

        # Cell 00: code cell header from Claude
        nb.cells.append(nbformat.v4.new_code_cell(source=cell00_src))

        # Cells 01..: placeholders matching source cell_type
        structure_items: List[Tuple[int, str]] = []
        for i in range(1, len(src_cells)):
            ctype = src_cells[i].get("cell_type", "code")
            title = ""
            if ctype == "code":
                nb.cells.append(_make_empty_code_cell_with_header(i, title=title))
            else:
                # keep markdown as markdown placeholder
                nb.cells.append(nbformat.v4.new_markdown_cell(source=f"<!-- Cell {i:02d} placeholder -->\n"))
            structure_items.append((i, title))

        # Save globals
        global LAST_STRUCTURE_ITEMS, LAST_SKELETON_NOTEBOOK_PATH
        LAST_STRUCTURE_ITEMS = structure_items
        LAST_SKELETON_NOTEBOOK_PATH = str(out_path)

    else:
        # Fallback: old behavior (Structure parsing) if no source provided
        raise RuntimeError("source_notebook_obj is required in cloner mode to match cell count exactly.")

    # Write
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nbformat.write(nb, str(out_path))

    if do_preview:
        print(f"[INFO] Skeleton written: {out_path} (cells={len(nb.cells)})")

    return nb


# ----------------------------
# Example usage (commented out)
# ----------------------------
# nb_obj = build_and_save_skeleton_from_cell00(
#     cell00_json=LAST_CLAUDE_CELL00_JSON,
#     out_path=Path("034_monitor_policy_daily.ipynb"),
#     do_preview=True,
# )
# preview_cell_sources(nb_obj, n_cells=2)


In [6]:
# ============================================================
# Cell 5 — Phase 1 Orchestrator (Generate Skeleton)
# ============================================================
# Overview:
# - Phase 1 end-to-end flow (function-only; UI is built in Cell 4 later):
#     1) Validate inputs (source notebook, target_domain, out_path)
#     2) Normalize SPEC_TEXT (fallback if empty)
#     3) (Optional) Retrieve Skill Pack
#     4) Call Claude to generate Cell 00 (CODE cell comment header + Structure)
#     5) Parse Structure and assemble skeleton notebook deterministically
#     6) Save skeleton .ipynb
#
# Inputs / Outputs:
# Inputs (globals set by UI in Cell 4):
# - LAST_SOURCE_NOTEBOOK_OBJ (required)
# - LAST_TARGET_DOMAIN (required)
# - LAST_OUT_NOTEBOOK_PATH (required)
# - LAST_SPEC_TEXT (optional)
# - LAST_SKILL_PACK (optional)
# - enable_skills_chk / skill_topk_slider / preview_skeleton_chk / preview_skills_chk (optional widgets)
#
# Outputs (globals):
# - LAST_CLAUDE_CELL00_JSON
# - LAST_STRUCTURE_ITEMS (set inside build_and_save_skeleton_from_cell00)
# - LAST_SKELETON_NOTEBOOK_PATH (set inside build_and_save_skeleton_from_cell00)
#
# Notes:
# - Deterministic: Claude writes Cell 00 only; Python assembles everything else.
# - UI wiring is intentionally NOT done here (Cell 4 handles buttons).
# ============================================================

from typing import Any, Dict, List, Optional
from pathlib import Path
from datetime import datetime

# ----------------------------
# Logging helper (never silent)
# ----------------------------
def _log_local(msg: str, level: str = "INFO"):
    if "_log" in globals() and callable(globals().get("_log")):
        _log(msg, level=level)
    elif "ui_log" in globals() and callable(globals().get("ui_log")):
        ui_log(msg, level=level)
    else:
        print(f"[{level}] {msg}")

def _now_str() -> str:
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ----------------------------
# Default SPEC builder
# ----------------------------
def build_default_spec_text(target_domain: str) -> str:
    return f"""
Clone the source notebook with minimal, domain-only edits.

TARGET_DOMAIN = {target_domain}

Rules:
- Keep the pipeline identical.
- Only modify domain-specific parts.
- Do not refactor or reorder cells.

Allowed changes:
- Monitoring Targets query/filter
- Event normalization tags and entity type
- Notebook title and output filename
""".strip()

# ----------------------------
# Input validation
# ----------------------------
def validate_phase1_inputs() -> None:
    if LAST_SOURCE_NOTEBOOK_OBJ is None:
        raise RuntimeError("LAST_SOURCE_NOTEBOOK_OBJ is None. Load/select a source notebook in the UI (Cell 4).")
    if not LAST_TARGET_DOMAIN:
        raise RuntimeError("LAST_TARGET_DOMAIN is empty. Set target domain in the UI (Cell 4) and apply inputs.")
    if LAST_OUT_NOTEBOOK_PATH is None:
        raise RuntimeError("LAST_OUT_NOTEBOOK_PATH is None. Set output path in the UI (Cell 4) and apply inputs.")
    if "generate_cell00_json" not in globals():
        raise RuntimeError("Cell 2 not executed: generate_cell00_json is missing.")
    if "build_and_save_skeleton_from_cell00" not in globals():
        raise RuntimeError("Cell 3 not executed: build_and_save_skeleton_from_cell00 is missing.")

# ----------------------------
# Phase 1 runner
# ----------------------------
def run_phase1_generate_skeleton(
    *,
    force_reselect_skills: bool = True,
    max_tokens_cell00: int = 20000,
    do_preview: Optional[bool] = None,
) -> Dict[str, Any]:
    """
    Run Phase 1 and update globals.
    Returns the assembled skeleton notebook object (dict).
    """
    validate_phase1_inputs()

    target_domain = LAST_TARGET_DOMAIN
    out_path = Path(LAST_OUT_NOTEBOOK_PATH)
    source_obj = LAST_SOURCE_NOTEBOOK_OBJ

    # Preview toggle: prefer widget if present
    if do_preview is None:
        do_preview = bool(preview_skeleton_chk.value) if "preview_skeleton_chk" in globals() else True

    # SPEC: use provided, else default
    spec_text = (LAST_SPEC_TEXT or "").strip()
    if not spec_text:
        spec_text = build_default_spec_text(target_domain)
        _log_local("SPEC_TEXT empty → using default spec.", level="INFO")

    # Skills: optional
    enable_skills = bool(enable_skills_chk.value) if "enable_skills_chk" in globals() else False
    top_k = int(skill_topk_slider.value) if "skill_topk_slider" in globals() else 5

    skill_pack: List[Dict[str, Any]] = []
    if enable_skills:
        if force_reselect_skills or globals().get("LAST_SKILL_PACK") is None:
            skill_pack = retrieve_skill_pack(
                spec_text=spec_text,
                source_notebook_obj=source_obj,
                target_domain=target_domain,
                top_k=top_k,
            )
            _log_local(f"Skill injection enabled: {len(skill_pack)} skills (top_k={top_k})", level="INFO")
            if "preview_skills_chk" in globals() and bool(preview_skills_chk.value):
                summarize_skill_pack(skill_pack)
        else:
            skill_pack = globals().get("LAST_SKILL_PACK") or []
            _log_local(f"Reusing Skill Pack: {len(skill_pack)} skills", level="INFO")
    else:
        _log_local("Skill injection disabled.", level="INFO")

    # Call Claude: generate Cell 00 JSON (code header style)
    _log_local(f"Calling Claude for Cell 00 (domain={target_domain}, out={out_path.name})", level="INFO")
    cell00_json = generate_cell00_json(
        target_domain=target_domain,
        out_notebook_name=out_path.name,
        spec_text=spec_text,
        source_notebook_obj=source_obj,
        skill_pack=skill_pack,
        max_tokens=max_tokens_cell00,
    )

    global LAST_CLAUDE_CELL00_JSON
    LAST_CLAUDE_CELL00_JSON = cell00_json
    _log_local("Received Cell 00 JSON.", level="INFO")

    # Assemble + save skeleton
    nb_obj = build_and_save_skeleton_from_cell00(
        cell00_json=cell00_json,
        out_path=out_path,
        do_preview=bool(do_preview),
        source_notebook_obj=source_obj,   # ★追加：source基準でセル数固定
    )

    _log_local(f"Phase 1 complete. Skeleton saved: {out_path}", level="INFO")
    _log_local(f"Timestamp: {_now_str()}", level="INFO")
    return nb_obj

def log_anywhere(msg: str, level: str = "INFO"):
    if "ui_log" in globals() and callable(ui_log):
        ui_log(msg, level=level)
    else:
        print(f"[{level}] {msg}")


In [7]:
# ============================================================
# Cell 6a — Phase 2 Builder (plan -> per-cell apply; keep=copy, edit=base64)
# ============================================================

from typing import Any, Dict, List, Optional, Tuple
from pathlib import Path
import json, re, base64

def _log_local(msg: str, level: str = "INFO"):
    if "_log" in globals() and callable(globals().get("_log")):
        _log(msg, level=level)
    elif "ui_log" in globals() and callable(globals().get("ui_log")):
        ui_log(msg, level=level)
    else:
        print(f"[{level}] {msg}")

# ---------- JSON extraction/parsing helpers ----------
def strip_code_fences_maybe(text: str) -> str:
    t = (text or "").strip()
    if t.startswith("```"):
        lines = t.splitlines()
        if len(lines) >= 2 and lines[-1].strip().startswith("```"):
            return "\n".join(lines[1:-1]).strip()
        return "\n".join(lines[1:]).strip()
    return t

def extract_first_json_object(text: str) -> str:
    t = strip_code_fences_maybe(text)
    start = t.find("{")
    if start < 0:
        return t
    depth, in_str, esc = 0, False, False
    for i in range(start, len(t)):
        ch = t[i]
        if in_str:
            if esc:
                esc = False
            elif ch == "\\":
                esc = True
            elif ch == '"':
                in_str = False
            continue
        else:
            if ch == '"':
                in_str = True
                continue
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return t[start:i+1]
    return t[start:]

def try_parse_json_loose(raw: str) -> Dict[str, Any]:
    s = extract_first_json_object(raw).strip()
    s = re.sub(r",\s*([}\]])", r"\1", s)  # trailing comma rescue
    return json.loads(s)

def clear_outputs_and_counts(nb: dict) -> dict:
    """
    Ensure notebook is clean before saving:
    - code cell outputs cleared
    - execution_count None
    Returns nb (mutates in-place for simplicity).
    """
    for c in nb.get("cells", []):
        if c.get("cell_type") == "code":
            c["execution_count"] = None
            c["outputs"] = []
    return nb

# ---------- notebook cell utils ----------
def cell_source_str(cell: dict) -> str:
    src = cell.get("source", "")
    return "".join(src) if isinstance(src, list) else str(src)

def set_cell_source(cell: dict, new_source: str) -> None:
    cell["source"] = new_source.splitlines(keepends=True)

def decode_b64_utf8(b64: str) -> str:
    return base64.b64decode(b64.encode("ascii")).decode("utf-8")

def shallow_clone_cell(cell: dict) -> dict:
    # safe clone (no outputs)
    c = json.loads(json.dumps(cell))
    if c.get("cell_type") == "code":
        c["outputs"] = []
        c["execution_count"] = None
    return c

# ---------- Step 1: PLAN ----------
def build_plan_system_prompt() -> str:
    return """
You are a senior Python/Jupyter engineer.
OUTPUT STRICT JSON ONLY (no markdown/fences/commentary).

Task:
For each provided SOURCE cell, decide if it can be kept as-is for TARGET_DOMAIN
or needs domain-only edits.

Hard rules:
- Output must be valid JSON.
- All string fields must be single-line (no literal newlines).
- Decision must be "keep" or "edit".
""".strip()

def build_plan_user_prompt(
    *,
    target_domain: str,
    allowlist: List[int],
    spec_text: str,
    source_nb: dict,
    max_chars_per_cell: int = 2200,
) -> str:
    schema = """
SCHEMA:
{
  "plan_id": "<string>",
  "target_domain": "<string>",
  "allowlist": [<int>, ...],
  "cells": [
    {"cell_index": <int>, "decision": "keep"|"edit", "reason": "<single-line>"}
  ]
}

Rules:
- Include EXACTLY one entry for every cell_index in allowlist.
- reason must be single-line (no newlines).
- Do NOT include any code or base64 in this step.
""".strip()

    chunks = []
    for i in allowlist:
        src = cell_source_str(source_nb["cells"][i])
        if len(src) > max_chars_per_cell:
            src = src[:max_chars_per_cell] + "\n# ...TRUNCATED..."
        chunks.append(f"\n### CELL_INDEX {i:02d}\n{src}\n")

    return f"""
TARGET_DOMAIN: {target_domain}

SPEC:
{spec_text}

ALLOWLIST:
{allowlist}

{schema}

SOURCE CELLS:
{''.join(chunks)}
""".strip()

def generate_plan(
    *,
    target_domain: Optional[str] = None,
    allowlist: Optional[List[int]] = None,
    max_tokens: int = 20000,
) -> Dict[str, Any]:
    if LAST_SOURCE_NOTEBOOK_OBJ is None:
        raise RuntimeError("LAST_SOURCE_NOTEBOOK_OBJ is None.")
    if "call_claude_messages_api" not in globals():
        raise RuntimeError("call_claude_messages_api missing (Cell 2).")

    target_domain = target_domain or LAST_TARGET_DOMAIN
    if not target_domain:
        raise RuntimeError("target_domain is empty.")

    source_nb = LAST_SOURCE_NOTEBOOK_OBJ

    # default: all code cells except cell00
    if allowlist is None:
        allowlist = [i for i,c in enumerate(source_nb.get("cells",[])) if i!=0 and c.get("cell_type")=="code"]

    spec_text = (LAST_SPEC_TEXT or "").strip()
    if not spec_text:
        spec_text = build_default_spec_text(target_domain) if "build_default_spec_text" in globals() else f"TARGET_DOMAIN={target_domain}"

    raw = call_claude_messages_api(
        system_prompt=build_plan_system_prompt(),
        user_prompt=build_plan_user_prompt(
            target_domain=target_domain,
            allowlist=allowlist,
            spec_text=spec_text,
            source_nb=source_nb,
        ),
        model=CLAUDE_MODEL,
        api_key=ANTHROPIC_API_KEY,
        max_tokens=max_tokens,
        temperature=0.1,
    )

    plan = try_parse_json_loose(raw)

    # validate
    if plan.get("target_domain") != target_domain:
        raise RuntimeError("PLAN target_domain mismatch.")
    if plan.get("allowlist") != allowlist:
        raise RuntimeError("PLAN allowlist mismatch (must echo exactly).")

    got = sorted(int(x["cell_index"]) for x in plan.get("cells", []))
    if got != sorted(allowlist):
        raise RuntimeError(f"PLAN must cover allowlist exactly. got={got}, expected={sorted(allowlist)}")

    global LAST_PATCH_PLAN
    LAST_PATCH_PLAN = plan

    edits = [c["cell_index"] for c in plan["cells"] if c["decision"]=="edit"]
    keeps = [c["cell_index"] for c in plan["cells"] if c["decision"]=="keep"]
    _log_local(f"Plan done. keep={keeps} edit={edits}", "INFO")
    return plan

# ---------- Step 2: single-cell EDIT generator (base64) ----------
def build_cell_edit_system_prompt() -> str:
    return """
You are a senior Python/Jupyter engineer.
OUTPUT STRICT JSON ONLY.

You will receive ONE source cell and TARGET_DOMAIN.
Return FULL replacement cell source as base64(utf-8).

Hard rules:
- Output valid JSON only.
- reason must be single-line.
- new_source_b64 must be single-line (no newlines).
- Minimal domain-only edits. No refactor. Keep logic identical.
""".strip()

def build_cell_edit_user_prompt(
    *,
    target_domain: str,
    cell_index: int,
    spec_text: str,
    source_cell_source: str,
    max_chars: int = 9000,
) -> str:
    if len(source_cell_source) > max_chars:
        source_cell_source = source_cell_source[:max_chars] + "\n# ...TRUNCATED..."

    schema = """
SCHEMA:
{
  "cell_index": <int>,
  "reason": "<single-line>",
  "new_source_b64": "<base64 utf-8, single-line>"
}
""".strip()

    return f"""
TARGET_DOMAIN: {target_domain}

SPEC:
{spec_text}

CELL_INDEX: {cell_index}

{schema}

SOURCE CELL:
{source_cell_source}
""".strip()

def generate_cell_edit_b64(
    *,
    cell_index: int,
    target_domain: Optional[str] = None,
    max_tokens: int = 20000,
) -> Dict[str, Any]:
    if LAST_SOURCE_NOTEBOOK_OBJ is None:
        raise RuntimeError("LAST_SOURCE_NOTEBOOK_OBJ is None.")
    target_domain = target_domain or LAST_TARGET_DOMAIN
    if not target_domain:
        raise RuntimeError("target_domain is empty.")

    spec_text = (LAST_SPEC_TEXT or "").strip()
    if not spec_text:
        spec_text = build_default_spec_text(target_domain) if "build_default_spec_text" in globals() else f"TARGET_DOMAIN={target_domain}"

    src = cell_source_str(LAST_SOURCE_NOTEBOOK_OBJ["cells"][cell_index])

    raw = call_claude_messages_api(
        system_prompt=build_cell_edit_system_prompt(),
        user_prompt=build_cell_edit_user_prompt(
            target_domain=target_domain,
            cell_index=cell_index,
            spec_text=spec_text,
            source_cell_source=src,
        ),
        model=CLAUDE_MODEL,
        api_key=ANTHROPIC_API_KEY,
        max_tokens=max_tokens,
        temperature=0.1,
    )
    _log_local(f"[cell {cell_index}] claude_return_type={type(raw).__name__}", "INFO")

    obj = try_parse_json_loose(raw)
    if int(obj.get("cell_index")) != int(cell_index):
        raise RuntimeError("CELL EDIT cell_index mismatch.")
    b64 = obj.get("new_source_b64", "")
    if not isinstance(b64, str) or not b64.strip():
        raise RuntimeError("CELL EDIT missing new_source_b64.")
    if "\n" in b64 or "\r" in b64:
        raise RuntimeError("CELL EDIT new_source_b64 must be single-line.")
    return obj

# ---------- Step 3: build a PATCH that 6b can apply (per run, one cell at a time) ----------
def build_patch_for_one_cell(
    *,
    target_domain: str,
    allowlist: List[int],
    cell_index: int,
    reason: str,
    new_source_b64: str,
) -> Dict[str, Any]:
    return {
        "patch_id": f"patch_{target_domain}_cell_{cell_index:02d}",
        "target_domain": target_domain,
        "allowed_cell_indices": allowlist,
        "changed_cells": [
            {"cell_index": int(cell_index), "reason": reason, "new_source_b64": new_source_b64}
        ]
    }

# ---------- Step 4: main loop builder ----------
def phase2_build_target_by_loop(
    *,
    plan: Optional[Dict[str, Any]] = None,
    apply_keep_immediately: bool = True,
    edit_one_by_one: bool = True,
    start_edit_from: Optional[int] = None,   # ★追加
) -> Dict[str, Any]:
    """
    Creates/updates an in-memory TARGET notebook object:
    - keep: copy source cell into target (python)
    - edit: generate base64 for that cell (one-by-one) and apply (python)
    Returns the updated target_nb object (dict).
    Also updates globals:
      - LAST_TARGET_NOTEBOOK_OBJ
      - LAST_PATCH_JSON (for last edited cell)
    """

    if LAST_SOURCE_NOTEBOOK_OBJ is None:
        raise RuntimeError("LAST_SOURCE_NOTEBOOK_OBJ is None.")
    if LAST_SKELETON_NOTEBOOK_PATH is None:
        raise RuntimeError("LAST_SKELETON_NOTEBOOK_PATH is None. Run Phase 1 first.")

    if "read_ipynb_from_path" not in globals():
        raise RuntimeError("read_ipynb_from_path missing (Cell 3).")

    target_domain = LAST_TARGET_DOMAIN
    if not target_domain:
        raise RuntimeError("LAST_TARGET_DOMAIN empty.")
    spec_text = (LAST_SPEC_TEXT or "").strip()
    if not spec_text:
        spec_text = build_default_spec_text(target_domain) if "build_default_spec_text" in globals() else f"TARGET_DOMAIN={target_domain}"

    source_nb = LAST_SOURCE_NOTEBOOK_OBJ
    target_nb = read_ipynb_from_path(Path(LAST_SKELETON_NOTEBOOK_PATH))

    plan = plan or globals().get("LAST_PATCH_PLAN")
    if not plan:
        raise RuntimeError("No plan provided and LAST_PATCH_PLAN is empty. Run generate_plan() first.")

    allowlist = plan["allowlist"]
    decisions = {int(c["cell_index"]): c for c in plan["cells"]}

    # 1) keep cells: copy from source to target
    if apply_keep_immediately:
        for idx, item in decisions.items():
            if item["decision"] == "keep":
                target_nb["cells"][idx] = shallow_clone_cell(source_nb["cells"][idx])
        _log_local("Applied KEEP cells by copying from source.", "INFO")

    # 2) edit cells: one-by-one
    edited_indices = [idx for idx,item in decisions.items() if item["decision"]=="edit"]
    edited_indices = sorted(edited_indices)

    if start_edit_from is not None:
        edited_indices = [i for i in edited_indices if i >= int(start_edit_from)]
        _log_local(f"Resuming edits from cell_index >= {start_edit_from}", "INFO")
        
    _log_local(f"EDIT cells to process: {edited_indices}", "INFO")

    if edit_one_by_one:
        # spec/skills are optional; keep them stable per run
        spec_text = (LAST_SPEC_TEXT or "").strip()
        if not spec_text:
            spec_text = build_default_spec_text(target_domain) if "build_default_spec_text" in globals() else f"TARGET_DOMAIN={target_domain}"
    
        skill_pack = globals().get("LAST_SKILL_PACK")
        if not isinstance(skill_pack, list):
            skill_pack = []
    
        # 2) edit cells: one-by-one
        for idx in edited_indices:
            _log_local(f"Editing cell {idx}...", "INFO")
        
            b64 = None
            new_src = None
        
            # 1) Try base64 first (default path)
            try:
                b64 = generate_cell_edit_b64_only_with_retry(
                    cell_index=idx,
                    target_domain=target_domain,
                    source_nb=source_nb,
                    current_target_nb=target_nb,
                    spec_text=spec_text,
                    skill_pack=skill_pack,
                    max_tokens=20000,
                    max_attempts=2,
                )
                new_src = decode_b64_utf8_strict(b64)
                _log_local(f"Cell {idx}: got base64 source. chars={len(new_src)} b64_len={len(b64)}", "INFO")
        
            except Exception as e:
                _log_local(f"[cell {idx}] base64 failed → fallback to text-only: {type(e).__name__}: {e}", "WARN")
        
                # 2) Fallback: text-only with tags
                new_src = generate_cell_edit_text_only_with_retry(
                    cell_index=idx,
                    target_domain=target_domain,
                    source_nb=source_nb,
                    spec_text=spec_text,
                    skill_pack=skill_pack,
                    max_tokens=20000,
                    max_attempts=2,
                )
                _log_local(f"Cell {idx}: got text-only source. chars={len(new_src)}", "INFO")
        
            # Apply to target
            target_nb["cells"][idx] = shallow_clone_cell(source_nb["cells"][idx])
            set_cell_source(target_nb["cells"][idx], new_src)
            _log_local(f"Edited cell {idx} applied to in-memory target.", "INFO")
        
            # Store last patch (if b64 missing, store empty or re-encode if you want)
            if b64 is None:
                # optional: store a base64 of text-only so patch shape stays consistent
                b64 = base64.b64encode(new_src.encode("utf-8")).decode("ascii")
        
            patch = build_patch_for_one_cell(
                target_domain=target_domain,
                allowlist=allowlist,
                cell_index=idx,
                reason=f"Domain-only edit for cell {idx}",
                new_source_b64=b64,
            )
            global LAST_PATCH_JSON
            LAST_PATCH_JSON = patch
            _log_local(f"Edited cell {idx} applied. decoded_chars={len(new_src)} b64_len={len(b64)}", "INFO")
    else:
        _log_local("edit_one_by_one=False: skipped edits.", "WARN")

    global LAST_TARGET_NOTEBOOK_OBJ
    LAST_TARGET_NOTEBOOK_OBJ = target_nb
    _log_local("Phase2 loop build complete. Stored LAST_TARGET_NOTEBOOK_OBJ.", "INFO")
    return target_nb

# ----------------------------
# Convenience runners (manual execution helpers)
# Put these at the END of Cell 6a
# ----------------------------

def run_phase2_plan() -> Dict[str, Any]:
    _log_local("Phase 2 PLAN starting...", "INFO")
    plan = generate_plan()
    _log_local("Phase 2 PLAN done.", "INFO")
    return plan

def run_phase2_build(plan: Optional[Dict[str, Any]] = None, edit_one_by_one: bool = True) -> Dict[str, Any]:
    _log_local("Phase 2 BUILD starting...", "INFO")
    plan = plan or globals().get("LAST_PATCH_PLAN")
    if not plan:
        raise RuntimeError("No plan found. Run run_phase2_plan() first.")
    nb = phase2_build_target_by_loop(plan=plan, apply_keep_immediately=True, edit_one_by_one=edit_one_by_one)
    _log_local("Phase 2 BUILD done.", "INFO")
    return nb

def run_phase2_save(out_path: Optional[str] = None) -> Path:
    _log_local("Phase 2 SAVE starting...", "INFO")
    p = save_target_notebook_obj(out_path=Path(out_path) if out_path else None)
    _log_local(f"Phase 2 SAVE done: {p}", "INFO")
    return p


def generate_cell_edit_b64_only(
    *,
    cell_index: int,
    target_domain: str,
    source_nb: dict,
    current_target_nb: dict,
    spec_text: str,
    skill_pack: Optional[list],
    max_tokens: int = 20000,
) -> str:
    """
    Ask Claude to output base64 wrapped in BEGIN_B64 / END_B64.
    """

    system_prompt = """
You are a senior Python/Jupyter engineer.

You MUST output BASE64 ONLY, wrapped exactly as follows:

BEGIN_B64
<base64 string here>
END_B64

Rules:
- Do NOT output anything outside BEGIN_B64 / END_B64
- No JSON
- No markdown
- No commentary
- Base64 must decode to UTF-8 text
- The decoded text must be the FULL replacement source for ONE code cell
""".strip()

    src_text = "".join(source_nb["cells"][cell_index].get("source", []))

    skills_text = (
        render_skill_pack_for_patch(skill_pack, max_items=6)
        if "render_skill_pack_for_patch" in globals()
        else "(no skills)"
    )

    user_prompt = f"""
TASK:
Edit cell {cell_index} only, to convert domain to: {target_domain}

Rules:
- Keep logic identical to SOURCE as much as possible.
- Only change domain-specific parts (targets filter/query, normalization tags, naming).
- Do not refactor.

SPEC:
{spec_text}

SKILLS:
{skills_text}

SOURCE CELL (index {cell_index}) ORIGINAL:
{src_text}

OUTPUT FORMAT (MANDATORY):
BEGIN_B64
<base64>
END_B64
""".strip()

    raw = call_claude_messages_api(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        model=CLAUDE_MODEL,
        api_key=ANTHROPIC_API_KEY,
        max_tokens=max_tokens,
        temperature=0.1,
    )

    b64 = extract_b64_between_tags(raw)

    decoded = decode_b64_to_text_strict(b64)
    if not decoded.strip():
        raise RuntimeError(f"Decoded cell source is empty for cell {cell_index}")

    return b64


def decode_b64_to_text(b64: str) -> str:
    import base64, re
    b64 = re.sub(r"\s+", "", b64 or "")
    return base64.b64decode(b64).decode("utf-8")

import base64, binascii, re, time

_B64_CHARS_RE = re.compile(r"[^A-Za-z0-9+/=]")

def normalize_base64_token(s: str) -> str:
    """
    - remove non-base64 chars
    - remove whitespace
    - add '=' padding if needed
    """
    s = (s or "").strip()
    s = re.sub(r"\s+", "", s)
    s = _B64_CHARS_RE.sub("", s)

    # pad to multiple of 4
    pad = (-len(s)) % 4
    if pad:
        s += "=" * pad
    return s


def _extract_b64_blob_stricter(text: str) -> str:
    """
    Robust base64 extractor.
    Accepts:
      - BEGIN_B64 ... END_B64
      - BEGIN_B64 + (no END_B64)  ← 今回のケース
      - plain base64 only
      - base64 with whitespace/newlines
    Returns: single-line base64 string (whitespace removed)
    """
    t = (text or "").strip()
    if not t:
        return ""

    # 1) Preferred: BEGIN_B64 ... END_B64
    m = re.search(r"BEGIN_B64\s*(.*?)\s*END_B64", t, flags=re.DOTALL)
    if m:
        blob = m.group(1)

    else:
        # 2) Fallback: BEGIN_B64 exists but END_B64 missing → take everything after BEGIN_B64
        m2 = re.search(r"BEGIN_B64\s*(.*)", t, flags=re.DOTALL)
        if m2:
            blob = m2.group(1)
        else:
            blob = t

    # 3) Remove non-base64 chars except whitespace, then remove whitespace
    blob = re.sub(r"[^A-Za-z0-9+/=\s]", " ", blob)
    blob = re.sub(r"\s+", "", blob)

    # 4) If we somehow captured too much, pick the longest base64-ish run
    cands = re.findall(r"[A-Za-z0-9+/=]{200,}", blob)
    if cands:
        blob = max(cands, key=len)

    # 5) Fix padding if needed (optional but helps "Incorrect padding")
    # base64 length should be multiple of 4
    pad = (-len(blob)) % 4
    if pad and pad < 4:
        blob = blob + ("=" * pad)

    return blob


def decode_b64_utf8_strict(b64: Any) -> str:
    """
    None-safe + whitespace除去 + padding補正 + UTF-8 strict decode.
    """
    if b64 is None:
        raise RuntimeError("base64 is None")

    s = re.sub(r"\s+", "", str(b64))
    if not s:
        raise RuntimeError("base64 is empty")

    # padding fix: base64 length must be multiple of 4
    pad = (-len(s)) % 4
    if pad:
        s = s + ("=" * pad)

    data = base64.b64decode(s, validate=False)

    try:
        return data.decode("utf-8")
    except UnicodeDecodeError as e:
        pos = e.start
        window = data[max(0, pos - 24): pos + 24].hex()
        raise RuntimeError(
            f"Base64 decoded bytes are not valid UTF-8: {e}. "
            f"pos={pos} bytes_window_hex={window}"
        )

def extract_b64_between_tags(raw: str) -> str:
    if raw is None:
        raise RuntimeError("Claude returned None")

    s = str(raw).strip()

    # 1) 正式: BEGIN ... END
    m = re.search(r"BEGIN_B64\s*([\s\S]+?)\s*END_B64", s)
    if m:
        b64 = m.group(1)
    else:
        # 2) fallback: BEGINだけある（END無し）→ BEGIN以降を拾う
        m2 = re.search(r"BEGIN_B64\s*([\s\S]+)$", s)
        if not m2:
            head = s[:300].replace("\n", "\\n")
            raise RuntimeError(f"BEGIN_B64 not found. raw_head={head}")
        b64 = m2.group(1)

    b64 = re.sub(r"\s+", "", b64)
    if not b64:
        raise RuntimeError("Extracted base64 is empty")

    # padding補正（Incorrect padding対策）
    pad = (-len(b64)) % 4
    if pad:
        b64 += "=" * pad

    return b64


def decode_b64_to_text_strict(b64: Any) -> str:
    """
    - None-safe
    - whitespace除去
    - padding補正
    - UTF-8 strict
    """
    if b64 is None:
        raise RuntimeError("decode_b64_to_text_strict: b64 is None")

    s = re.sub(r"\s+", "", str(b64))

    if not s:
        raise RuntimeError("decode_b64_to_text_strict: empty base64 string")

    # padding fix: base64 length must be multiple of 4
    pad = (-len(s)) % 4
    if pad:
        s = s + ("=" * pad)

    data = base64.b64decode(s, validate=False)

    try:
        return data.decode("utf-8")
    except UnicodeDecodeError as e:
        pos = e.start
        window = data[max(0, pos-24):pos+24].hex()
        raise RuntimeError(
            f"Base64 decoded bytes are not valid UTF-8: {e}. "
            f"pos={pos} bytes_window_hex={window}"
        )

def looks_truncated_python(code: str) -> bool:
    s = (code or "").rstrip()
    # ありがちな “途中で終わり” パターン
    if s.endswith(("get", "=", "(", "[", "{", ",", ":", "\\")):
        return True
    # 最後の行がインデント途中で終わってるのも怪しい
    last = s.splitlines()[-1] if s.splitlines() else ""
    if last.strip() and last.strip()[-1] in {",", "(", "[", "{", ":"}:
        return True
    return False

def python_syntax_ok(code: str) -> bool:
    try:
        compile(code, "<cell>", "exec")
        return True
    except SyntaxError:
        return False

def generate_cell_edit_b64_only_with_retry(
    *,
    cell_index: int,
    target_domain: str,
    source_nb: dict,
    current_target_nb: dict,  # 未使用でもOK（互換のため残す）
    spec_text: str,
    skill_pack: Optional[list],
    max_tokens: int = 20000,
    max_attempts: int = 4,
) -> str:
    """
    Ask Claude to output base64 wrapped in BEGIN_B64 / END_B64.
    Retry if:
    - raw is None
    - tags missing
    - base64 invalid / empty
    - decoded text not utf-8
    - decoded text empty
    - decoded text looks truncated / invalid python
    """

    system_prompt = """
You are a senior Python/Jupyter engineer.

Return EXACTLY 3 lines and nothing else:
LINE 1: BEGIN_B64
LINE 2: <base64 of UTF-8 full cell source, single line>
LINE 3: END_B64

No JSON. No markdown. No backticks. No commentary.
""".strip()

    src_text = "".join(source_nb["cells"][cell_index].get("source", []))

    skills_text = (
        render_skill_pack_for_patch(skill_pack, max_items=6)
        if "render_skill_pack_for_patch" in globals()
        else "(no skills)"
    )

    user_prompt = f"""
Edit cell {cell_index} only to convert domain to: {target_domain}

Rules:
- Keep logic identical to SOURCE as much as possible.
- Only change domain-specific parts (targets filter/query, normalization tags, naming).
- Do not refactor.

SPEC:
{spec_text}

SKILLS:
{skills_text}

SOURCE CELL (index {cell_index}) ORIGINAL:
{src_text}

OUTPUT FORMAT (MANDATORY):
BEGIN_B64
<base64>
END_B64
""".strip()
    user_prompt += "\n\nREMINDER: Output exactly 3 lines: BEGIN_B64 / <single-line base64> / END_B64."

    last_err = None
    last_raw_head = None

    for attempt in range(1, max_attempts + 1):
        raw = None
        try:
            raw = call_claude_messages_api(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                model=CLAUDE_MODEL,
                api_key=ANTHROPIC_API_KEY,
                max_tokens=max_tokens,
                temperature=0.1,
                timeout_s=600,
                max_retries=3,
            )

            if raw is None:
                raise RuntimeError("Claude returned None (timeout/network).")

            # ✅ ここが変更点：タグ間だけを抽出
            b64 = extract_b64_between_tags(raw)

            # ✅ decode & validate
            decoded = decode_b64_utf8_strict(b64)

            if not decoded.strip():
                raise RuntimeError("Decoded UTF-8 text is empty.")

            if "looks_truncated_python" in globals() and callable(globals().get("looks_truncated_python")):
                if looks_truncated_python(decoded):
                    raise RuntimeError("Decoded cell looks truncated; retrying.")

            if "python_syntax_ok" in globals() and callable(globals().get("python_syntax_ok")):
                if not python_syntax_ok(decoded):
                    raise RuntimeError("Decoded cell is not valid Python; retrying.")

            return b64

        except Exception as e:
            last_err = e
            last_raw_head = (str(raw) if raw is not None else "None")[:240].replace("\n", "\\n")
            _log_local(
                f"[cell {cell_index}] base64 attempt {attempt}/{max_attempts} failed: {type(e).__name__}: {e}",
                "WARN",
            )
            _log_local(f"[cell {cell_index}] raw_head={last_raw_head}", "WARN")
            time.sleep(1.5 * attempt)

    raise RuntimeError(
        f"Base64 decode failed for cell {cell_index} after {max_attempts} attempts. "
        f"last_err={last_err} raw_head={last_raw_head}"
    )
    
def generate_cell_edit_text_only_with_retry(
    *,
    cell_index: int,
    target_domain: str,
    source_nb: dict,
    spec_text: str,
    skill_pack: Optional[list],
    max_tokens: int = 20000,
    max_attempts: int = 3,
) -> str:
    """
    Claudeに「コード（生テキスト）だけ」を返させる。
    ただし安定のため BEGIN_CELL / END_CELL で囲ませて、その中身だけ抽出する。
    """

    system_prompt = """
You are a senior Python/Jupyter engineer.

Return ONLY the full Python code for ONE Jupyter code cell,
wrapped exactly as 3 blocks:

BEGIN_CELL
<python code here>
END_CELL

Rules:
- Output must include BOTH BEGIN_CELL and END_CELL.
- Do NOT add extra explanations, long headers, or "REPLACEMENT".
- Keep comments minimal. Prefer copying SOURCE and editing only domain tokens.
- No JSON, no base64, no markdown.
""".strip()

    src_text = "".join(source_nb["cells"][cell_index].get("source", []))
    skills_text = (
        render_skill_pack_for_patch(skill_pack, max_items=6)
        if "render_skill_pack_for_patch" in globals()
        else "(no skills)"
    )

    user_prompt = f"""
Edit cell {cell_index} only to convert domain to: {target_domain}

Rules:
- Keep logic identical to SOURCE as much as possible.
- Only change domain-specific parts (targets filter/query, normalization tags, naming).
- Do not refactor.
- Output must be the FULL replacement cell source (no truncation).

SPEC:
{spec_text}

SKILLS:
{skills_text}

SOURCE CELL:
{src_text}

OUTPUT FORMAT (MANDATORY):
BEGIN_CELL
<full cell source>
END_CELL
""".strip()

    def _extract_between_tags(raw: str) -> str:
        if raw is None:
            return ""
        s = str(raw)
        a = s.find("BEGIN_CELL")
        b = s.rfind("END_CELL")
        if a < 0 or b < 0 or b <= a:
            return ""
        return s[a + len("BEGIN_CELL") : b].strip()

    last_err = None
    for attempt in range(1, max_attempts + 1):
        raw = None
        try:
            raw = call_claude_messages_api(
                system_prompt=system_prompt,
                user_prompt=user_prompt,
                model=CLAUDE_MODEL,
                api_key=ANTHROPIC_API_KEY,
                max_tokens=max_tokens,
                temperature=0.1,
                timeout_s=600,
                max_retries=3,
            )

            text = _extract_between_tags(raw)

            if not text:
                raise RuntimeError("Missing BEGIN_CELL/END_CELL or empty body.")

            # 念のため fence を拒否（タグ方式でも混ざることがある）
            if "```" in text:
                raise RuntimeError("Markdown fence found in extracted text.")

            # 最低長
            if len(text) < 120:
                raise RuntimeError(f"Too short output (len={len(text)})")

            # 途中で切れてそうな典型パターン（末尾が変）
            if text.rstrip().endswith((",", "(", "[", "{", "\\", "=", "or", "and")):
                raise RuntimeError("Output looks truncated (bad ending token).")

            # 可能なら構文チェック（ある場合のみ）
            if "python_syntax_ok" in globals() and callable(globals().get("python_syntax_ok")):
                if not python_syntax_ok(text):
                    raise RuntimeError("Python syntax check failed; likely truncated.")

            return text

        except Exception as e:
            last_err = e
            head = (str(raw) if raw is not None else "None")[:240].replace("\n", "\\n")
            _log_local(f"[cell {cell_index}] text-only attempt {attempt}/{max_attempts} failed: {type(e).__name__}: {e}", "WARN")
            _log_local(f"[cell {cell_index}] raw_head={head}", "WARN")
            time.sleep(1.5 * attempt)

    raise RuntimeError(
        f"Text-only generation failed for cell {cell_index} after {max_attempts} attempts. last_err={last_err}"
    )

In [8]:
# ============================================================
# Cell 6b — Patch Applier + Verifier (Phase 2, BASE64 + source-clone-first)
# ============================================================

from typing import Any, Dict, List, Optional
from pathlib import Path
import json
import base64
import difflib
import nbformat

def apply_patch_to_notebook(skeleton_nb: dict, patch: dict) -> dict:
    nb2 = _clone_json(skeleton_nb)

    allowed = patch["allowed_cell_indices"]
    changed_cells = patch["changed_cells"]

    for ch in changed_cells:
        idx = int(ch["cell_index"])
        if idx not in allowed:
            raise RuntimeError(f"Patch attempts disallowed cell {idx}")
        if nb2["cells"][idx].get("cell_type") != "code":
            raise RuntimeError(f"Cell {idx} is not code")

        b64 = ch.get("new_source_b64", "")
        if not b64:
            raise RuntimeError(f"Missing new_source_b64 for cell {idx}")
        if "\n" in b64 or "\r" in b64:
            raise RuntimeError(f"new_source_b64 must be single-line for cell {idx}")

        new_source = base64.b64decode(b64.encode("ascii")).decode("utf-8")
        set_cell_source(nb2["cells"][idx], new_source)

    return clear_outputs_and_counts(nb2)

def save_target_notebook_obj(*, out_path: Optional[Path] = None) -> Path:
    if "LAST_TARGET_NOTEBOOK_OBJ" not in globals() or LAST_TARGET_NOTEBOOK_OBJ is None:
        raise RuntimeError("LAST_TARGET_NOTEBOOK_OBJ is None. Run Phase 2 BUILD first.")
    if out_path is None:
        out_path = globals().get("LAST_OUT_NOTEBOOK_PATH")
    if out_path is None:
        raise RuntimeError("out_path is None and LAST_OUT_NOTEBOOK_PATH is None.")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    nb_obj = LAST_TARGET_NOTEBOOK_OBJ

    # ★ここが今回の修正ポイント
    if "clear_outputs_and_counts" not in globals():
        raise RuntimeError("clear_outputs_and_counts is not defined. Define it in Cell 6a utilities.")
    clear_outputs_and_counts(nb_obj)

    nbformat.write(nbformat.from_dict(nb_obj), str(out_path))

    global LAST_FINAL_NOTEBOOK_PATH
    LAST_FINAL_NOTEBOOK_PATH = out_path
    _log_local(f"Saved target notebook: {out_path}", "INFO")
    return out_path

def _log_local(msg: str, level: str = "INFO"):
    if "_log" in globals() and callable(globals().get("_log")):
        _log(msg, level=level)
    elif "ui_log" in globals() and callable(globals().get("ui_log")):
        ui_log(msg, level=level)
    else:
        print(f"[{level}] {msg}")

def b64_decode_text(s: str) -> str:
    return base64.b64decode(s.encode("ascii")).decode("utf-8")

def _clone_json(obj: Any) -> Any:
    return json.loads(json.dumps(obj))

def clear_outputs(nb: dict) -> dict:
    nb2 = _clone_json(nb)
    for c in nb2.get("cells", []):
        if c.get("cell_type") == "code":
            c["execution_count"] = None
            c["outputs"] = []
    return nb2

def cell_src(cell: dict) -> str:
    src = cell.get("source", "")
    return "".join(src) if isinstance(src, list) else str(src)

def set_cell_src(cell: dict, src: str):
    cell["source"] = src.splitlines(keepends=True)

def unified_diff(a: str, b: str, fromfile: str, tofile: str, n: int = 3) -> str:
    return "".join(difflib.unified_diff(
        a.splitlines(keepends=True),
        b.splitlines(keepends=True),
        fromfile=fromfile,
        tofile=tofile,
        n=n,
    ))

def run_phase2_apply_verify_save(
    *,
    out_path: Optional[Path] = None,
    do_print_log: bool = True,
) -> Path:

    _pp_banner("Phase 2b START")

    try:

        if LAST_PATCH_JSON is None:
            raise RuntimeError("LAST_PATCH_JSON is None. Run Phase 2a first.")
        if LAST_SKELETON_NOTEBOOK_PATH is None:
            raise RuntimeError("LAST_SKELETON_NOTEBOOK_PATH is None. Run Phase 1 first.")
        if LAST_SOURCE_NOTEBOOK_OBJ is None:
            raise RuntimeError("LAST_SOURCE_NOTEBOOK_OBJ is None. Load source notebook first.")
    
        patch = LAST_PATCH_JSON
        out_path = Path(out_path or LAST_OUT_NOTEBOOK_PATH)
    
        source_nb = clear_outputs(LAST_SOURCE_NOTEBOOK_OBJ)
        skeleton_nb = clear_outputs(read_ipynb_from_path(Path(LAST_SKELETON_NOTEBOOK_PATH)))
    
        if len(source_nb.get("cells", [])) != len(skeleton_nb.get("cells", [])):
            raise RuntimeError(
                f"Cell count mismatch source({len(source_nb.get('cells', []))}) vs skeleton({len(skeleton_nb.get('cells', []))}). "
                "Phase 1 structure must match source for clone-first strategy."
            )
    
        # 1) Start from skeleton, but fill Cell 01..N by copying source byte-identically
        target_nb = _clone_json(skeleton_nb)
        for i in range(1, len(target_nb["cells"])):
            # Keep cell types identical to skeleton; copy only source for code cells
            if target_nb["cells"][i].get("cell_type") != source_nb["cells"][i].get("cell_type"):
                raise RuntimeError(f"cell_type mismatch at index {i} between source and skeleton.")
            if target_nb["cells"][i].get("cell_type") == "code":
                set_cell_src(target_nb["cells"][i], cell_src(source_nb["cells"][i]))
    
        # 2) Apply patch decisions (edit only; keep means leave source as-is)
        allowed = list(map(int, patch["allowed_cell_indices"]))
        changed_cells = patch["changed_cells"]
    
        edited_indices: List[int] = []
        for ch in changed_cells:
            idx = int(ch["cell_index"])
            if idx not in allowed:
                raise RuntimeError(f"Patch tries to touch disallowed cell {idx}")
            decision = ch.get("decision")
            if decision == "edit":
                new_src = b64_decode_text(ch["new_source_b64"])
                if target_nb["cells"][idx].get("cell_type") != "code":
                    raise RuntimeError(f"Can only edit code cells. index {idx} is {target_nb['cells'][idx].get('cell_type')}")
                set_cell_src(target_nb["cells"][idx], new_src)
                edited_indices.append(idx)
            elif decision == "keep":
                pass
            else:
                raise RuntimeError(f"Invalid decision at cell {idx}: {decision}")
    
        target_nb = clear_outputs(target_nb)
    
        # 3) Verify invariants:
        # - allowlist outside: must be identical to source
        # - within allowlist: keep => identical to source, edit => may differ
        diffs_outside = []
        diffs_keep = []
        diffs_edit = []
    
        allowset = set(allowed)
        editset = set(edited_indices)
    
        for i in range(len(source_nb["cells"])):
            # metadata strict
            if (source_nb["cells"][i].get("metadata") or {}) != (target_nb["cells"][i].get("metadata") or {}):
                raise RuntimeError(f"metadata changed at index {i}")
            if source_nb["cells"][i].get("cell_type") != target_nb["cells"][i].get("cell_type"):
                raise RuntimeError(f"cell_type changed at index {i}")
    
            if target_nb["cells"][i].get("cell_type") != "code":
                continue
    
            s_src = cell_src(source_nb["cells"][i])
            s_tgt = cell_src(target_nb["cells"][i])
    
            if i not in allowset:
                if s_src != s_tgt:
                    diffs_outside.append(i)
            else:
                if i in editset:
                    if s_src != s_tgt:
                        diffs_edit.append(i)
                else:
                    # keep decision implied: must be identical to source
                    if s_src != s_tgt:
                        diffs_keep.append(i)
    
        if diffs_outside:
            raise RuntimeError(f"Changes detected OUTSIDE allowlist (not allowed): {diffs_outside}")
        if diffs_keep:
            raise RuntimeError(f"Cells marked keep (or not edited) changed unexpectedly: {diffs_keep}")
    
        # 4) Save
        out_path.parent.mkdir(parents=True, exist_ok=True)
        nbformat.write(nbformat.from_dict(target_nb), str(out_path))
    
        global LAST_FINAL_NOTEBOOK_PATH, LAST_PATCH_LOG
        LAST_FINAL_NOTEBOOK_PATH = out_path
    
        # Make a small python-generated log
        lines = []
        lines.append("# Patch Log (Python-generated)")
        lines.append(f"- patch_id: {patch.get('patch_id')}")
        lines.append(f"- target_domain: {patch.get('target_domain')}")
        lines.append(f"- allowed_cell_indices: {allowed}")
        lines.append(f"- edited_cell_indices: {sorted(editset)}")
        lines.append("")
        for idx in sorted(editset):
            before = cell_src(source_nb["cells"][idx])
            after = cell_src(target_nb["cells"][idx])
            lines.append(f"## Cell index {idx:02d}")
            lines.append("```diff")
            lines.append(unified_diff(before, after, f"before:{idx:02d}", f"after:{idx:02d}").rstrip("\n"))
            lines.append("```")
            lines.append("")
        LAST_PATCH_LOG = "\n".join(lines) + "\n"
    
        _log_local(f"Phase 2b complete. Saved: {out_path}", "INFO")
        _log_local(f"Edited cells: {sorted(editset)}", "INFO")
    
        if do_print_log:
            print(LAST_PATCH_LOG)
    
        return out_path
        out = _existing_run_phase2_apply_verify_save(out_path=out_path, do_print_log=do_print_log)

        _pp_ok("Phase 2b", extra=f"- saved: {out}")
        return out

    except Exception as e:
        _pp_exc(e, context="Phase 2b")
        raise

In [9]:
# ============================================================
# Cell 4 — UI (ipywidgets)
# ============================================================
# Overview:
# - Human-in-the-loop control surface for Phase 1/Phase 2.
# - Select SOURCE_NOTEBOOK via:
#     (A) Upload widget (recommended), OR
#     (B) Local directory dropdown (optional convenience)
# - Collect TARGET_DOMAIN, OUT_NOTEBOOK path, preview toggles, and Skill controls.
# - This cell wires buttons that call:
#     - Phase 1: run_phase1_generate_skeleton()
#     - Phase 2a: generate_phase2_patch_json()
#     - Phase 2b: run_phase2_apply_verify_save()
#
# Notes:
# - This cell is intended to run LAST (after Cells 1–3, 5–6b define functions).
# ============================================================

from typing import List, Optional
import ipywidgets as widgets
from IPython.display import display
from pathlib import Path

# ----------------------------
# Logger
# ----------------------------
log_out = widgets.Output()

def ui_log(msg: str, level: str = "INFO"):
    with log_out:
        print(f"[{level}] {msg}")

def ui_clear_log():
    log_out.clear_output()

# ----------------------------
# Guard helpers (fail loud in UI)
# ----------------------------
def _require(name: str):
    if name not in globals() or globals().get(name) is None:
        raise RuntimeError(f"Missing required symbol: {name}. Run the defining cell first.")

def _safe_project_dir_default() -> Path:
    # Prefer PROJECT_DIR if defined; else current working dir; else home.
    if "PROJECT_DIR" in globals() and globals().get("PROJECT_DIR"):
        try:
            return Path(globals()["PROJECT_DIR"]).expanduser().resolve()
        except Exception:
            pass
    try:
        return Path.cwd().resolve()
    except Exception:
        return Path.home().resolve()

def _safe_path(s: str) -> Path:
    return Path(s).expanduser().resolve()

def list_ipynb_files(root: Path) -> List[Path]:
    if not root.exists():
        return []
    return sorted([p for p in root.glob("*.ipynb") if p.is_file()])

# ----------------------------
# Globals setters
# ----------------------------
def _apply_inputs_to_globals():
    global LAST_TARGET_DOMAIN, LAST_OUT_NOTEBOOK_PATH, LAST_SPEC_TEXT
    LAST_TARGET_DOMAIN = target_domain_text.value.strip() or None

    out_s = out_path_text.value.strip()
    LAST_OUT_NOTEBOOK_PATH = Path(out_s) if out_s else None

    spec_s = spec_text_area.value.strip()
    LAST_SPEC_TEXT = spec_s or None

def _apply_skill_pack_to_globals():
    global LAST_SKILL_PACK

    if not enable_skills_chk.value:
        LAST_SKILL_PACK = []
        ui_log("Skill injection disabled (LAST_SKILL_PACK set to empty).", level="INFO")
        return

    _require("retrieve_skill_pack")

    top_k = int(skill_topk_slider.value)
    pack = retrieve_skill_pack(
        spec_text=(spec_text_area.value.strip() or None),
        source_notebook_obj=globals().get("LAST_SOURCE_NOTEBOOK_OBJ"),
        target_domain=(target_domain_text.value.strip() or None),
        top_k=top_k,
    )
    LAST_SKILL_PACK = pack
    ui_log(f"Selected Skill Pack: {len(pack)} skills (top_k={top_k}).", level="INFO")

    if preview_skills_chk.value:
        _require("summarize_skill_pack")
        summarize_skill_pack(pack)

# ----------------------------
# Source notebook selection (Upload / Local)
# ----------------------------
source_mode = widgets.ToggleButtons(
    options=[("Upload .ipynb", "upload"), ("Pick local .ipynb", "local")],
    value="upload",
    description="Source:",
)

upload_widget = widgets.FileUpload(
    accept=".ipynb",
    multiple=False,
    description="Upload",
)

local_dir_text = widgets.Text(
    value=str(_safe_project_dir_default()),
    description="Dir:",
    layout=widgets.Layout(width="600px"),
)

refresh_local_btn = widgets.Button(description="Refresh list")

local_file_dropdown = widgets.Dropdown(
    options=[],
    description="File:",
    layout=widgets.Layout(width="800px"),
)

def _refresh_local_dropdown():
    root = _safe_path(local_dir_text.value)
    files = list_ipynb_files(root)
    local_file_dropdown.options = [str(p) for p in files]
    local_file_dropdown.value = str(files[0]) if files else None

def _set_source_from_upload(change=None):
    global LAST_SOURCE_NOTEBOOK_PATH, LAST_SOURCE_NOTEBOOK_OBJ

    if not upload_widget.value:
        return

    _require("read_ipynb_from_bytes")

    uploaded = list(upload_widget.value.values())
    if not uploaded:
        return

    item = uploaded[0]
    content = item.get("content", None)

    # Try multiple keys for filename across environments
    name = (
        item.get("metadata", {}).get("name")
        or item.get("name")
        or "uploaded.ipynb"
    )

    if content is None:
        ui_log("Upload content missing.", level="WARN")
        return

    try:
        nb_obj = read_ipynb_from_bytes(content)
    except Exception as e:
        ui_log(f"Failed to parse uploaded ipynb: {type(e).__name__}: {e}", level="ERROR")
        return

    LAST_SOURCE_NOTEBOOK_PATH = None
    LAST_SOURCE_NOTEBOOK_OBJ = nb_obj
    ui_log(f"Loaded source notebook from upload: {name} (cells={len(nb_obj.get('cells', []))})", level="INFO")

def _set_source_from_local_selection(change=None):
    global LAST_SOURCE_NOTEBOOK_PATH, LAST_SOURCE_NOTEBOOK_OBJ

    if not local_file_dropdown.value:
        return

    _require("read_ipynb_from_path")

    p = Path(local_file_dropdown.value)
    try:
        nb_obj = read_ipynb_from_path(p)
    except Exception as e:
        ui_log(f"Failed to read ipynb: {p} error={type(e).__name__}: {e}", level="ERROR")
        return

    LAST_SOURCE_NOTEBOOK_PATH = p
    LAST_SOURCE_NOTEBOOK_OBJ = nb_obj
    ui_log(f"Loaded source notebook from disk: {p.name} (cells={len(nb_obj.get('cells', []))})", level="INFO")

# init local list
_refresh_local_dropdown()

# Handlers
upload_widget.observe(_set_source_from_upload, names="value")
refresh_local_btn.on_click(lambda _: _refresh_local_dropdown())
local_file_dropdown.observe(_set_source_from_local_selection, names="value")

# Mode panel
source_panel_upload = widgets.VBox([
    widgets.HTML("<b>Upload mode</b>: choose a .ipynb from your machine"),
    upload_widget,
])

source_panel_local = widgets.VBox([
    widgets.HTML("<b>Local mode</b>: pick a .ipynb from a directory in this runtime"),
    widgets.HBox([local_dir_text, refresh_local_btn]),
    local_file_dropdown,
])

source_panel_stack = widgets.Stack([source_panel_upload, source_panel_local])
source_panel_stack.selected_index = 0

def _on_source_mode_change(change):
    mode = change["new"]
    source_panel_stack.selected_index = 0 if mode == "upload" else 1
    ui_log(f"Switched source mode to: {mode}", level="INFO")

source_mode.observe(_on_source_mode_change, names="value")

# ----------------------------
# Target / output / spec / preview
# ----------------------------
target_domain_text = widgets.Text(value="policy", description="Domain:", layout=widgets.Layout(width="300px"))
out_path_text = widgets.Text(value="034_monitor_policy_daily.ipynb", description="Out:", layout=widgets.Layout(width="600px"))

spec_text_area = widgets.Textarea(
    value="",
    description="Spec:",
    layout=widgets.Layout(width="900px", height="160px"),
    placeholder="(optional) Paste a human spec here. Leave empty to rely on pipeline invariants.",
)

preview_skeleton_chk = widgets.Checkbox(value=True, description="Preview skeleton summary")

# ----------------------------
# Skills controls
# ----------------------------
enable_skills_chk = widgets.Checkbox(value=True, description="Enable Skill injection")
skill_topk_slider = widgets.IntSlider(value=5, min=0, max=12, step=1, description="Top-K:")
preview_skills_chk = widgets.Checkbox(value=False, description="Print selected skills")

# ----------------------------
# Buttons: Apply / Skills / Phase 1/2 / Log
# ----------------------------
apply_inputs_btn = widgets.Button(description="Apply inputs", button_style="info")
select_skills_btn = widgets.Button(description="Select Skill Pack")
clear_log_btn = widgets.Button(description="Clear log")

phase1_btn = widgets.Button(description="Phase 1: Generate skeleton", button_style="success")
phase2_plan_btn  = widgets.Button(description="Phase 2: PLAN (keep/edit)", button_style="warning")
phase2_build_btn = widgets.Button(description="Phase 2: BUILD (loop)", button_style="warning")
phase2_save_btn  = widgets.Button(description="Phase 2: SAVE", button_style="success")
phase2_all_btn   = widgets.Button(description="Phase 2: RUN ALL (plan→build→save)", button_style="danger")


def _on_apply_inputs_clicked(_):
    _apply_inputs_to_globals()
    ui_log("Applied Domain / Out / Spec to globals.", level="INFO")
    ui_log(f"LAST_TARGET_DOMAIN={globals().get('LAST_TARGET_DOMAIN')}", level="INFO")
    ui_log(f"LAST_OUT_NOTEBOOK_PATH={globals().get('LAST_OUT_NOTEBOOK_PATH')}", level="INFO")
    ui_log(f"LAST_SPEC_TEXT={'(set)' if globals().get('LAST_SPEC_TEXT') else '(empty)'}", level="INFO")

def _on_select_skills_clicked(_):
    _apply_inputs_to_globals()
    _apply_skill_pack_to_globals()

def _on_clear_log_clicked(_):
    ui_clear_log()

def _on_phase1_clicked(_):
    try:
        _require("run_phase1_generate_skeleton")
        _apply_inputs_to_globals()
        ui_log("Phase 1 starting...", level="INFO")
        run_phase1_generate_skeleton(force_reselect_skills=True)
        ui_log("Phase 1 done.", level="INFO")
    except Exception as e:
        ui_log(f"Phase 1 failed: {type(e).__name__}: {e}", level="ERROR")
        raise

def _on_phase2_plan_clicked(_):
    try:
        _require("run_phase2_plan")
        _apply_inputs_to_globals()
        ui_log("Phase 2 PLAN starting...", level="INFO")
        plan = run_phase2_plan()
        ui_log(f"Phase 2 PLAN done. edits={[c['cell_index'] for c in plan['cells'] if c['decision']=='edit']}", level="INFO")
    except Exception as e:
        ui_log(f"Phase 2 PLAN failed: {type(e).__name__}: {e}", level="ERROR")
        raise

def _on_phase2_build_clicked(_):
    try:
        _require("run_phase2_build")
        _apply_inputs_to_globals()
        ui_log("Phase 2 BUILD starting...", level="INFO")
        run_phase2_build(plan=globals().get("LAST_PATCH_PLAN"), edit_one_by_one=True)
        ui_log("Phase 2 BUILD done.", level="INFO")
    except Exception as e:
        ui_log(f"Phase 2 BUILD failed: {type(e).__name__}: {e}", level="ERROR")
        raise

def _on_phase2_save_clicked(_):
    try:
        _require("run_phase2_save")
        _apply_inputs_to_globals()
        ui_log("Phase 2 SAVE starting...", level="INFO")
        p = run_phase2_save()
        ui_log(f"Phase 2 SAVE done: {p}", level="INFO")
    except Exception as e:
        ui_log(f"Phase 2 SAVE failed: {type(e).__name__}: {e}", level="ERROR")
        raise

def _on_phase2_all_clicked(_):
    try:
        _require("run_phase2_plan")
        _require("run_phase2_build")
        _require("run_phase2_save")
        _apply_inputs_to_globals()

        ui_log("Phase 2 RUN ALL starting...", level="INFO")
        plan = run_phase2_plan()
        run_phase2_build(plan=plan, edit_one_by_one=True)
        p = run_phase2_save()
        ui_log(f"Phase 2 RUN ALL done: {p}", level="INFO")
    except Exception as e:
        ui_log(f"Phase 2 RUN ALL failed: {type(e).__name__}: {e}", level="ERROR")
        raise


apply_inputs_btn.on_click(_on_apply_inputs_clicked)
select_skills_btn.on_click(_on_select_skills_clicked)
clear_log_btn.on_click(_on_clear_log_clicked)

phase1_btn.on_click(_on_phase1_clicked)
phase2_plan_btn.on_click(_on_phase2_plan_clicked)
phase2_build_btn.on_click(_on_phase2_build_clicked)
phase2_save_btn.on_click(_on_phase2_save_clicked)
phase2_all_btn.on_click(_on_phase2_all_clicked)


# ----------------------------
# Render UI (single panel)
# ----------------------------
ui = widgets.VBox([
    widgets.HTML("<h3>037 Monitor Notebook Cloner — Control Panel</h3>"),
    source_mode,
    source_panel_stack,
    widgets.HBox([target_domain_text, out_path_text]),
    spec_text_area,
    widgets.HBox([preview_skeleton_chk]),
    widgets.HTML("<hr><b>Skills</b>"),
    widgets.HBox([enable_skills_chk, skill_topk_slider, preview_skills_chk]),
    widgets.HBox([apply_inputs_btn, select_skills_btn, clear_log_btn]),
    widgets.HTML("<hr><b>Run</b>"),
    widgets.VBox([
    widgets.HBox([phase1_btn]),
    widgets.HBox([phase2_plan_btn, phase2_build_btn, phase2_save_btn]),
    widgets.HBox([phase2_all_btn]),
    ]),
    widgets.HTML("<hr><b>Logs</b>"),
    log_out,
])

display(ui)
ui_log("UI ready. 1) Select source notebook, 2) Set Domain/Out, 3) Apply inputs, 4) (Optional) Select Skill Pack, 5) Run Phase 1 → 2a → 2b.", level="INFO")
